# Notebook for setting up and analysing CICE6 
### This will serve as my python notebook for sharing
After establishing that CICE6 'gx1' and 'gx3' grids run through to completion on the 'vanilla' setup (see https://github.com/dpath2o/AFIM/blob/main/MODELS/cice.org). The 'tx1' runs to completion but appears to not read-in the forcing data as all the data variables in 'history' file output are 'nan'. At this stage I think the way forward is:
1. create a cartesian grid using an example from either ACCESS-OM2 or other sources online
2. create bathymetry on the same grid scale
3. create forcing on the same grid scale
4. create initial conditions on the same grid scale

In [ ]:
import os
import warnings
import cartopy
import cartopy.crs       as ccrs
import matplotlib.pyplot as plt
import matplotlib        as mpl
import xarray            as xr
import pandas            as pd
import numpy             as np
import metpy.calc        as mpc
from xgcm                    import Grid
from matplotlib.ticker       import MaxNLocator
from datetime                import datetime, timedelta
from mpl_toolkits.axes_grid1 import make_axes_locatable,Divider,Size
from cartopy.mpl.gridliner   import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
%matplotlib inline
warnings.filterwarnings("ignore")

## The 'vanilla' CICE6 run output
These three history files were output from the three grids available from CICE6 (https://cice-consortium-cice.readthedocs.io/en/master/index.html)

In [ ]:
gx1 = xr.open_dataset('/Users/dpath2o/cice-dirs/runs/sandbox_gx1/history/iceh_ic.2005-01-01-03600.nc')
tx1 = xr.open_dataset('/Users/dpath2o/cice-dirs/runs/sandbox_tx1/history/iceh_ic.2005-01-01-03600.nc')
gx3 = xr.open_dataset('/Users/dpath2o/cice-dirs/runs/sandbox_gx3/history/iceh_ic.2005-01-01-03600.nc')

## ACCESS-OM2 grid file (courtesy of Paul)

In [ ]:
nc1 = xr.open_dataset('/Users/dpath2o/PHD/MODELS/CICE_runs/grids/cice_1deg_grid_from_paul.nc')

## Following this ( https://gallery.pangeo.io/repos/xgcm/xgcm-examples/03_MOM6.html ) example

In [ ]:
dataurl = 'http://35.188.34.63:8080/thredds/dodsC/OM4p5/ocean_monthly_z.200301-200712.nc4'
om4p5   = xr.open_dataset(f'{dataurl}',
                          chunks={'time':1, 'z_l': 1}, drop_variables=['average_DT',
                                                                       'average_T1',
                                                                       'average_T2'],
                          engine='netcdf4')
#grid = Grid(om4p5, coords={'X': {'center': 'xh', 'outer': 'xq'},
#                            'Y': {'center': 'yh', 'outer': 'yq'},
#                            'Z': {'inner': 'z_l', 'outer': 'z_i'} }, periodic=['X'
om4p5

## My attempt at creating a CICE6-compatible grid from the MOM6

In [ ]:
Gout = xr.Dataset(coords=dict(TLON=(["nj","ni"], -om4p5.geolon.values),
                              TLAT=(["nj","ni"], om4p5.geolat.values),
                              ULON=(["ny","nx"],-om4p5.geolon_c.values),
                              ULAT=(["ny","nx"],om4p5.geolat.values)),
                  attrs=dict(description="0.5 degree c-grid from MOM6"))
Gout.to_netcdf("/Users/dpath2o/cice-dirs/input/CICE_data/grid/gx0p5/grid_px0p5.nc")

## Plotting the grids

In [ ]:
fig = plt.figure(figsize=(15,12))
ax  = plt.axes(projection=ccrs.PlateCarree())
lons = -om4p5.isel(xh=slice(1,100),yh=slice(1,200)).geolon
lats = om4p5.isel(xh=slice(1,100),yh=slice(1,200)).geolat
plt.plot(lons,lats,transform=ccrs.PlateCarree(),marker='.',linestyle='none',color='k')
ax.coastlines()
tit_str = 'MOM6 grid'.format()
plt.title(tit_str)
plt.show()
plt.close(fig)

In [ ]:
fig = plt.figure(figsize=(15,12))
ax  = plt.axes(projection=ccrs.PlateCarree())
lons = tx1.isel(nj=slice(1,100),ni=slice(1,200)).TLON
lats = tx1.isel(nj=slice(1,100),ni=slice(1,200)).TLAT
plt.plot(lons,lats,transform=ccrs.PlateCarree(),marker='.',linestyle='none',color='k')
ax.coastlines()
tit_str = 'CICE6-TX1 -- Binary grid'.format()
plt.title(tit_str)
plt.show()
plt.close(fig)

In [ ]:
fig = plt.figure(figsize=(15,12))
ax  = plt.axes(projection=ccrs.PlateCarree())
lons = -np.rad2deg(nc1.isel(nx=slice(1,100),ny=slice(1,200)).tlon)
lats = np.rad2deg(nc1.isel(nx=slice(1,100),ny=slice(1,200)).tlat)
plt.plot(lons,lats,transform=ccrs.PlateCarree(),marker='.',linestyle='none',color='k')
ax.coastlines()
tit_str = 'ACCESS-OM grid'.format()
plt.title(tit_str)
plt.show()
plt.close(fig)

In [ ]:
om0p25_grd = xr.open_dataset("/Users/dpath2o/DATA/ACCESS-OM2/grids/ocean_grid_025.nc")
fig = plt.figure(figsize=(15,12))
ax  = plt.axes(projection=ccrs.PlateCarree())
lons = -om0p25_grd.isel(xt_ocean=slice(1,100),yt_ocean=slice(1,200)).geolon_t
lats = om0p25_grd.isel(xt_ocean=slice(1,100),yt_ocean=slice(1,200)).geolat_t
plt.plot(lons,lats,transform=ccrs.PlateCarree(),marker='.',linestyle='none',color='k')
ax.coastlines()
tit_str = 'COSIMA grid'.format()
plt.title(tit_str)
plt.show()
plt.close(fig)
om0p25_grd

In [ ]:
print(om0p25_grd.xt_ocean.min())
print(om0p25_grd.xt_ocean.max())
print(np.ptp(om0p25_grd.xt_ocean.values,axis=0))
print(om0p25_grd.geolon_t.min())
print(om0p25_grd.geolon_t.max())
print(np.ptp(om0p25_grd.geolon_t.values,axis=1))
print(om0p25_grd.geolon_t.values[0,:])
print(om0p25_grd.geolon_c.min())
print(om0p25_grd.geolon_c.max())
print(np.ptp(om0p25_grd.geolon_c.values,axis=1))
print(om0p25_grd.geolon_c.values[0,:])

In [ ]:
om0p25_grd_yu_xu_mask = xr.open_dataset("/Users/dpath2o/DATA/ACCESS-OM2/grids/om2_p1deg_basinmask_yu_xu.nc")
om0p25_grd_yu_xu_mask

In [ ]:
t2 = xr.open_dataset("/Users/dpath2o/DATA/ERA5/single-levels/2t_era5_oper_sfc_20211201-20211231.nc")
t2